# Stock Markets Analytics Zoomcamp (2026 Cohort)
## Module 3 Homework: Time Series Modeling, Handcrafted Rules, and Decision Trees

This notebook contains the complete solution for **Module 3 Homework (2026 Cohort)** based on [`cohorts/2026/homework3.md`](../cohorts/2026/homework3.md).

### Objectives:
1. **Question 1**: Generate dummies for Month and Week-of-Month (`<month>_w<week>`), perform correlation analysis with `is_positive_growth_30d_future`, and find the highest absolute correlation.
2. **Question 2**: Define handcrafted prediction rules based on Macro indicators (`DGS10`, `DGS5`, `FEDFUNDS`) and calculate their precision on the TEST dataset.
3. **Question 3**: Train a 10-level Decision Tree Classifier (`clf10`) on TRAIN + VALIDATION data, predict on the full dataset, and count records in the TEST set where only the Decision Tree is correct while all handcrafted rules fail.
4. **Question 4**: Perform hyperparameter tuning on `max_depth` (1 to 12) to identify the optimal tree depth that maximizes precision on the TEST dataset.
5. **Question 5**: Exploratory analysis proposing high-value alternative data sources to enhance stock growth prediction models.

### 0. Environment Setup & Data Loading

We load the combined stocks dataset for 33 large-cap stocks (`stocks_df_combined_2026_09_18.parquet.brotli`), filter for the historical window $\ge$ 2000-01-01 (~25 years of daily trading observations), and establish numerical and categorical feature sets.

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import precision_score, accuracy_score

# Load full parquet dataset
parquet_file = "stocks_df_combined_2026_09_18.parquet.brotli"
df_full = pd.read_parquet(parquet_file)
print(f"Original full dataset shape: {df_full.shape}")

# Filter to >= 2000-01-01 (approx. 25 years of daily observations)
df = df_full[df_full.Date >= '2000-01-01'].copy().reset_index(drop=True)
del df_full
print(f"Filtered dataset shape (Date >= 2000-01-01): {df.shape}")

# Feature definitions
GROWTH = [g for g in df.keys() if (g.startswith('growth_')) and ('future' not in g)]
OHLCV = ['Open', 'High', 'Low', 'Close', 'Adj Close_x', 'Volume']

# Create ln_volume feature
df['ln_volume'] = df.Volume.replace(0, np.nan).fillna(1e-9).apply(lambda x: np.log(x))
CUSTOM_NUMERICAL = ['SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative', 'volatility', 'ln_volume']

TECHNICAL_INDICATORS = [
    'adx', 'adxr', 'apo', 'aroon_1', 'aroon_2', 'aroonosc',
    'bop', 'cci', 'cmo', 'dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext',
    'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix',
    'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo',
    'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk',
    'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr',
    'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase',
    'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine',
    'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice'
]

TECHNICAL_PATTERNS = [g for g in df.keys() if 'cdl' in g]
MACRO = ['gdppot_us_yoy', 'gdppot_us_qoq', 'cpi_core_yoy', 'cpi_core_mom', 'FEDFUNDS', 'DGS1', 'DGS5', 'DGS10']

NUMERICAL = GROWTH + TECHNICAL_INDICATORS + TECHNICAL_PATTERNS + CUSTOM_NUMERICAL + MACRO
TO_PREDICT = [g for g in df.keys() if 'future' in g]

print(f"Total numerical features: {len(NUMERICAL)}")
print(f"Target variables to predict: {TO_PREDICT}")

Original full dataset shape: (240759, 203)
Filtered dataset shape (Date >= 2000-01-01): (202292, 204)
Total numerical features: 184
Target variables to predict: ['growth_future_30d', 'is_positive_growth_30d_future']


---
### Question 1: Dummies for Month and Week-of-Month

**What is the ABSOLUTE CORRELATION VALUE of the most correlated dummy variable `<month>_w<week_of_month>` with the binary outcome `is_positive_growth_30d_future`?**

#### Instructions:
1. Compute the week of the month using the formula: `(d.day - 1) // 7 + 1`.
2. Create a string combining month name and week of the month (e.g., `'October_w1'`).
3. Add `month_wom` to the categorical feature list:
   - `'Month'`
   - `'Weekday'`
   - `'Ticker'`
   - `'ticker_type'`
   - `'month_wom'`
4. Generate dummy variables using `pandas.get_dummies()`.
5. Compute correlation with `is_positive_growth_30d_future`.
6. Filter for `month_wom` dummy variables and find the highest absolute correlation value rounded to **three decimal places**.

In [2]:
# Step 1: Week of month formula
df['wom'] = (df['Date'].dt.day - 1) // 7 + 1

# Step 2: Combine Month name and Week of Month
df['month_wom'] = df['Date'].dt.month_name() + '_w' + df['wom'].astype(str)
print("Sample month_wom values:", df[['Date', 'month_wom']].head(5).to_dict(orient='records'))

# Step 3: Define categorical features list
df['Weekday'] = df['Weekday'].astype(str)
df['Month'] = df['Month'].dt.month.astype(str)
CATEGORICAL = ['Month', 'Weekday', 'Ticker', 'ticker_type', 'month_wom']

# Step 4: Generate dummy variables
dummy_variables = pd.get_dummies(df[CATEGORICAL], dtype='int32')
print(f"Total dummy variables generated: {dummy_variables.shape[1]}")

month_wom_dummies = [col for col in dummy_variables.columns if col.startswith('month_wom_')]
print(f"Total month_wom dummy variables: {len(month_wom_dummies)}")

# Step 5: Compute correlation with target variable is_positive_growth_30d_future
corrs = {}
target = df['is_positive_growth_30d_future']
for col in month_wom_dummies:
    corrs[col] = dummy_variables[col].corr(target)

corr_df = pd.DataFrame(list(corrs.items()), columns=['feature', 'corr'])
corr_df['abs_corr'] = corr_df['corr'].abs()
corr_sorted = corr_df.sort_values(by='abs_corr', ascending=False).reset_index(drop=True)

print("\nTop 10 most correlated month_wom dummy variables:")
print(corr_sorted.head(10).to_string(index=False))

top_feature = corr_sorted.iloc[0]['feature']
top_abs_corr = corr_sorted.iloc[0]['abs_corr']
print(f"\nHighest correlated feature: {top_feature}")
print(f"Absolute correlation: {top_abs_corr:.6f}")
print(f"Rounded to 3 decimal places: {top_abs_corr:.3f}")

Sample month_wom values: [{'Date': Timestamp('2000-01-03 00:00:00'), 'month_wom': 'January_w1'}, {'Date': Timestamp('2000-01-04 00:00:00'), 'month_wom': 'January_w1'}, {'Date': Timestamp('2000-01-05 00:00:00'), 'month_wom': 'January_w1'}, {'Date': Timestamp('2000-01-06 00:00:00'), 'month_wom': 'January_w1'}, {'Date': Timestamp('2000-01-07 00:00:00'), 'month_wom': 'January_w1'}]
Total dummy variables generated: 115
Total month_wom dummy variables: 60

Top 10 most correlated month_wom dummy variables:
              feature      corr  abs_corr
 month_wom_October_w4  0.024584  0.024584
month_wom_November_w3  0.023251  0.023251
month_wom_February_w1 -0.020861  0.020861
 month_wom_January_w2 -0.019846  0.019846
 month_wom_January_w5 -0.019792  0.019792
month_wom_November_w2  0.019103  0.019103
 month_wom_January_w3 -0.018537  0.018537
month_wom_February_w2 -0.018340  0.018340
 month_wom_January_w4 -0.018265  0.018265
 month_wom_October_w3  0.017856  0.017856

Highest correlated feature: mont

### Question 1 Answer:
> **0.025**
>
> The most correlated dummy variable is `month_wom_October_w4` with an absolute correlation of **`0.024584`**, which rounds to **`0.025`** (3 decimal places).

---
### Temporal Split: Train (70%), Validation (15%), Test (15%)

We split the historical observations by time interval order:
- **Train period**: 2000-01-03 to 2018-09-13 (136,842 observations, ~67.6%)
- **Validation period**: 2018-09-14 to 2022-09-15 (32,287 observations, ~16.0%)
- **Test period**: 2022-09-16 to 2026-09-18 (33,163 observations, ~16.4%)

In [3]:
# Concatenate dummy variables to dataframe
df_with_dummies = pd.concat([df, dummy_variables], axis=1)

def temporal_split(df, min_date, max_date, train_prop=0.7, val_prop=0.15, test_prop=0.15):
    train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train_prop)
    val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val_prop)
    
    split_labels = []
    for date in df['Date']:
        if date <= train_end:
            split_labels.append('train')
        elif date <= val_end:
            split_labels.append('validation')
        else:
            split_labels.append('test')
            
    df['split'] = split_labels
    return df

df_with_dummies = temporal_split(
    df_with_dummies,
    min_date=df_with_dummies.Date.min(),
    max_date=df_with_dummies.Date.max()
)

new_df = df_with_dummies.copy()
print("Split proportions:")
print(new_df['split'].value_counts(normalize=True))
print("\nSplit row counts:")
print(new_df['split'].value_counts())

Split proportions:
split
train         0.676458
test          0.163936
validation    0.159606
Name: proportion, dtype: float64

Split row counts:
split
train         136842
test           33163
validation     32287
Name: count, dtype: int64


---
### Question 2: Define New "Hand" Rules on Macro and Technical Indicator Variables

**What is the precision score for the best of the NEW predictions (`pred3` or `pred4`), rounded to 3 digits after the comma?**

#### Instructions:
1. Define two new handcrafted rules based on branches leading to positive predictions:
   - `pred3_manual_dgs10_5`: `(DGS10 <= 4.5) & (DGS5 <= 4)`
   - `pred4_manual_dgs10_fedfunds`: `(DGS10 > 4) & (FEDFUNDS <= 4.795)`
2. Apply these rules alongside `pred0`, `pred1`, and `pred2`:
   - `pred0_manual_cci`: `(cci > 200)`
   - `pred1_manual_prev_g1`: `(growth_30d > 1)`
   - `pred2_manual_prev_g1_and_snp`: `(growth_30d > 1) & (growth_snp500_30d > 1)`
3. Compute the precision score on the **TEST** dataset (`TP / (TP + FP)`) for the rule that makes positive predictions. Round to **three decimal places**.

In [4]:
# Baseline handcrafted rules from lecture
new_df['pred0_manual_cci'] = (new_df['cci'] > 200).astype(int)
new_df['pred1_manual_prev_g1'] = (new_df['growth_30d'] > 1).astype(int)
new_df['pred2_manual_prev_g1_and_snp'] = ((new_df['growth_30d'] > 1) & (new_df['growth_snp500_30d'] > 1)).astype(int)

# New handcrafted rules for Question 2
new_df['pred3_manual_dgs10_5'] = ((new_df['DGS10'] <= 4.5) & (new_df['DGS5'] <= 4)).astype(int)
new_df['pred4_manual_dgs10_fedfunds'] = ((new_df['DGS10'] > 4) & (new_df['FEDFUNDS'] <= 4.795)).astype(int)

PREDICTIONS = [
    'pred0_manual_cci',
    'pred1_manual_prev_g1',
    'pred2_manual_prev_g1_and_snp',
    'pred3_manual_dgs10_5',
    'pred4_manual_dgs10_fedfunds'
]

# Track correct predictions
for pred in PREDICTIONS:
    part1 = pred.split('_')[0]
    new_df[f'is_correct_{part1}'] = (new_df[pred] == new_df['is_positive_growth_30d_future']).astype(int)

# Evaluate precision on the TEST dataset
test_mask = (new_df['split'] == 'test')
y_test = new_df.loc[test_mask, 'is_positive_growth_30d_future']

summary_q2 = []
for p_col in PREDICTIONS:
    p_test = new_df.loc[test_mask, p_col]
    pos_mask = (p_test == 1)
    n_pos = pos_mask.sum()
    if n_pos > 0:
        tp = ((p_test == 1) & (y_test == 1)).sum()
        fp = ((p_test == 1) & (y_test == 0)).sum()
        prec = tp / (tp + fp)
        summary_q2.append({
            'Rule': p_col,
            'Positive Predictions': n_pos,
            'TP': tp,
            'FP': fp,
            'Precision': prec,
            'Precision (3 decimals)': f"{prec:.3f}"
        })

df_summary_q2 = pd.DataFrame(summary_q2)
print("Evaluation of Hand Rules on TEST Dataset:")
print(df_summary_q2.to_string(index=False))

# Compare new rules pred3 vs pred4
p3_prec = df_summary_q2.loc[df_summary_q2['Rule'] == 'pred3_manual_dgs10_5', 'Precision'].values[0]
p4_prec = df_summary_q2.loc[df_summary_q2['Rule'] == 'pred4_manual_dgs10_fedfunds', 'Precision'].values[0]
best_new_prec = max(p3_prec, p4_prec)
print(f"\nPrecision pred3: {p3_prec:.6f} -> rounded: {p3_prec:.3f}")
print(f"Precision pred4: {p4_prec:.6f} -> rounded: {p4_prec:.3f}")
print(f"Best NEW prediction precision: {best_new_prec:.3f}")

Evaluation of Hand Rules on TEST Dataset:
                        Rule  Positive Predictions    TP   FP  Precision Precision (3 decimals)
            pred0_manual_cci                   921   520  401   0.564604                  0.565
        pred1_manual_prev_g1                 19158 11086 8072   0.578662                  0.579
pred2_manual_prev_g1_and_snp                 15244  8728 6516   0.572553                  0.573
        pred3_manual_dgs10_5                 15910  9351 6559   0.587744                  0.588
 pred4_manual_dgs10_fedfunds                 15921  8001 7920   0.502544                  0.503

Precision pred3: 0.587744 -> rounded: 0.588
Precision pred4: 0.502544 -> rounded: 0.503
Best NEW prediction precision: 0.588


### Question 2 Answer:
> **0.588**
>
> Between the two new rules, `pred3_manual_dgs10_5` achieved a precision score of **`0.587744`**, which rounds to **`0.588`** (3 decimal places), whereas `pred4_manual_dgs10_fedfunds` achieved **`0.503`**.

---
### Question 3: Unique Correct Predictions from a 10-Level Decision Tree Classifier (`pred5_clf_10`)

**What is the total number of records in the TEST dataset where the new prediction `pred5_clf_10` is correct, while all 'hand' rule predictions (`pred0` to `pred4`) are incorrect?**

#### Instructions:
1. Initialize `DecisionTreeClassifier(max_depth=10, random_state=42)` for reproducibility.
2. Fit the classifier on the combined **TRAIN + VALIDATION** dataset after imputing $\pm\infty$ and NaNs with 0.
3. Predict on the entire dataset and store predictions in column `pred5_clf_10`.
4. Create a boolean/integer column `only_pred5_is_correct` where:
   - `pred5_clf_10` matches the true label.
   - All other hand rules (`pred0` to `pred4`) fail to match the true label.
5. Filter for the **TEST** dataset and report the count of records where `only_pred5_is_correct == 1`.

In [5]:
DUMMIES = [c for c in dummy_variables.columns]
features_list = NUMERICAL + DUMMIES
to_predict = 'is_positive_growth_30d_future'

# Prepare Train + Validation combined dataset
train_val_mask = new_df['split'].isin(['train', 'validation'])
X_train_val = new_df.loc[train_val_mask, features_list].copy()
y_train_val = new_df.loc[train_val_mask, to_predict].copy()

# Data cleaning: Replace +/-inf and NaNs with 0
X_train_val.replace([np.inf, -np.inf], np.nan, inplace=True)
X_train_val.fillna(0, inplace=True)

# Step 1: Train DecisionTreeClassifier(max_depth=10, random_state=42)
clf10 = DecisionTreeClassifier(max_depth=10, random_state=42)
clf10.fit(X_train_val, y_train_val)

# Step 2: Predict on the entire dataset
X_all = new_df[features_list].copy()
X_all.replace([np.inf, -np.inf], np.nan, inplace=True)
X_all.fillna(0, inplace=True)

new_df['pred5_clf_10'] = clf10.predict(X_all)

# Step 3: Identify records where only pred5 is correct
y_true = new_df[to_predict]
pred5_correct = (new_df['pred5_clf_10'] == y_true)
hand_rules_incorrect = (
    (new_df['pred0_manual_cci'] != y_true) &
    (new_df['pred1_manual_prev_g1'] != y_true) &
    (new_df['pred2_manual_prev_g1_and_snp'] != y_true) &
    (new_df['pred3_manual_dgs10_5'] != y_true) &
    (new_df['pred4_manual_dgs10_fedfunds'] != y_true)
)

new_df['only_pred5_is_correct'] = (pred5_correct & hand_rules_incorrect).astype(int)

# Count unique correct predictions on the TEST set
test_subset = new_df[new_df['split'] == 'test']
unique_correct_test_count = test_subset['only_pred5_is_correct'].sum()

print(f"Total TEST observations: {len(test_subset)}")
print(f"Records where pred5_clf_10 is correct: {(test_subset['pred5_clf_10'] == test_subset[to_predict]).sum()}")
print(f"Records where ONLY pred5_clf_10 is correct (pred0..pred4 all incorrect): {unique_correct_test_count}")
print(f"Proportion of TEST records uniquely solved by clf10: {unique_correct_test_count / len(test_subset):.4%}")

Total TEST observations: 33163
Records where pred5_clf_10 is correct: 15831
Records where ONLY pred5_clf_10 is correct (pred0..pred4 all incorrect): 1243
Proportion of TEST records uniquely solved by clf10: 3.7482%


In [6]:
# Advanced (Optional): Generalized function using pandas row apply
def is_uniquely_correct(row, target_pred, other_preds, label_col):
    if row[target_pred] != row[label_col]:
        return 0
    for other in other_preds:
        if row[other] == row[label_col]:
            return 0
    return 1

hand_cols = ['pred0_manual_cci', 'pred1_manual_prev_g1', 'pred2_manual_prev_g1_and_snp', 'pred3_manual_dgs10_5', 'pred4_manual_dgs10_fedfunds']
# Verifying row apply produces identical result on sample
sample_test = test_subset.head(100).copy()
sample_test['check_unique'] = sample_test.apply(
    lambda r: is_uniquely_correct(r, 'pred5_clf_10', hand_cols, to_predict),
    axis=1
)
assert (sample_test['check_unique'] == sample_test['only_pred5_is_correct']).all()
print("Verification passed: Generalized row evaluation matches vectorized implementation.")

Verification passed: Generalized row evaluation matches vectorized implementation.


### Question 3 Answer:
> **1243**
>
> There are **1,243 records** in the TEST dataset where the 10-level Decision Tree (`pred5_clf_10`) correctly predicts future positive growth, while all five handcrafted rules (`pred0` through `pred4`) fail.

---
### Question 4: Hyperparameter Tuning for a Decision Tree

**What is the optimal tree depth (from 1 to 12) for a DecisionTreeClassifier?**

#### Instructions:
1. Iterate `max_depth` from 1 to 12.
2. For each depth, train `DecisionTreeClassifier(max_depth=depth, random_state=42)` on TRAIN + VALIDATION.
3. Calculate the precision score on the **TEST** dataset (`TP / (TP + FP)`).
4. Identify the optimal `best_max_depth` where TEST precision is maximized.
5. Retrain using `best_max_depth` and compare performance with earlier models.

In [7]:
# Prepare Test set features and ground truth
X_test = new_df.loc[test_mask, features_list].copy()
y_test = new_df.loc[test_mask, to_predict].copy()
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.fillna(0, inplace=True)

# Also prepare Validation set to monitor generalization / overfitting
val_mask = (new_df['split'] == 'validation')
X_val = new_df.loc[val_mask, features_list].copy()
y_val = new_df.loc[val_mask, to_predict].copy()
X_val.replace([np.inf, -np.inf], np.nan, inplace=True)
X_val.fillna(0, inplace=True)

tuning_results = []
best_max_depth = None
highest_test_precision = -1.0

for depth in range(1, 13):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train_val, y_train_val)
    
    test_preds = clf.predict(X_test)
    val_preds = clf.predict(X_val)
    
    prec_test = precision_score(y_test, test_preds)
    prec_val = precision_score(y_val, val_preds)
    acc_test = accuracy_score(y_test, test_preds)
    pos_preds_count = (test_preds == 1).sum()
    
    tuning_results.append({
        'max_depth': depth,
        'Test Precision': prec_test,
        'Val Precision': prec_val,
        'Test Accuracy': acc_test,
        'Positive Predictions': pos_preds_count
    })
    
    if prec_test > highest_test_precision:
        highest_test_precision = prec_test
        best_max_depth = depth

df_tuning = pd.DataFrame(tuning_results)
print("Hyperparameter Tuning Results (max_depth 1 to 12):")
print(df_tuning.to_string(index=False))

print(f"\nOptimal max_depth: {best_max_depth}")
print(f"Highest precision on TEST dataset: {highest_test_precision:.5f} ({highest_test_precision:.3f})")

Hyperparameter Tuning Results (max_depth 1 to 12):
 max_depth  Test Precision  Val Precision  Test Accuracy  Positive Predictions
         1        0.570093       0.569982       0.570093                 33163
         2        0.622534       0.566144       0.516811                 11760
         3        0.571969       0.575027       0.491813                 14263
         4        0.629175       0.567083       0.481923                  6677
         5        0.599973       0.561494       0.475047                  7487
         6        0.606205       0.563068       0.472846                  6704
         7        0.621072       0.559281       0.478700                  6682
         8        0.621785       0.564551       0.499020                  9410
         9        0.611179       0.560662       0.490607                  9053
        10        0.591215       0.558362       0.477371                  8628
        11        0.602754       0.561491       0.476707                  7552
 

In [8]:
# Train best tuned DecisionTree model (max_depth = 4)
clf_best = DecisionTreeClassifier(max_depth=best_max_depth, random_state=42)
clf_best.fit(X_train_val, y_train_val)

new_df['pred6_clf_best'] = clf_best.predict(X_all)
y_test_pred6 = new_df.loc[test_mask, 'pred6_clf_best']
prec_best = precision_score(y_test, y_test_pred6)

print(f"Tuned Model (pred6_clf_best, depth={best_max_depth}) Test Precision: {prec_best:.5f}")
print("\nComparison of all models on TEST dataset:")
for pred_name in PREDICTIONS + ['pred5_clf_10', 'pred6_clf_best']:
    preds = new_df.loc[test_mask, pred_name]
    pos_mask = (preds == 1)
    if pos_mask.sum() > 0:
        p = ((preds == 1) & (y_test == 1)).sum() / pos_mask.sum()
        print(f"  {pred_name:30s} | Precision: {p:.5f}")

Tuned Model (pred6_clf_best, depth=4) Test Precision: 0.62917

Comparison of all models on TEST dataset:
  pred0_manual_cci               | Precision: 0.56460
  pred1_manual_prev_g1           | Precision: 0.57866
  pred2_manual_prev_g1_and_snp   | Precision: 0.57255
  pred3_manual_dgs10_5           | Precision: 0.58774
  pred4_manual_dgs10_fedfunds    | Precision: 0.50254
  pred5_clf_10                   | Precision: 0.59121
  pred6_clf_best                 | Precision: 0.62917


### Question 4 Answer:
> **4**
>
> The optimal tree depth is **`max_depth = 4`**, reaching the highest test precision score of **`0.62917`** (`62.9%`). This tuned tree significantly outperforms the unpruned 10-level tree (`0.59121`) as well as all handcrafted rules (`0.503` – `0.588`).

---
### [EXPLORATORY] Question 5: What data is missing?

**Prompt**:
> *Now that you have gained insights from the correlation analysis and Decision Tree results regarding the most influential variables, suggest new indicators you would like to include in the dataset and explain your reasoning. Alternatively, propose a completely different approach based on your intuition, specifying data sources for the 33 large-cap stocks across India, EU, and the US.*

---

#### 1. Market Microstructure & Order Book Flow (High-Frequency Liquidity)
* **What is missing**:
  Current features rely on daily aggregated OHLCV and TA-Lib indicators calculated at end-of-day. They lack information about intraday limit order book dynamics, liquidity imbalances, and institutional execution footprints.
* **Proposed Indicators**:
  - **Order Book Imbalance (OBI)**: $\frac{\text{BidVol} - \text{AskVol}}{\text{BidVol} + \text{AskVol}}$ at Top 5 levels of the book.
  - **Volume-Weighted Bid-Ask Spread (VWBAS)**: Measures effective transaction costs and liquidity withdrawal ahead of major volatility shifts.
  - **Institutional Trade Flow & Block Trade Ratios**: Distinguishes retail flow from algorithmic block execution.
* **Data Sources**:
  - [Databento](https://databento.com/) (Normalized Level 2 / Level 3 market depth).
  - Exchange direct feeds: Nasdaq TotalView (US), Euronext Best of Book (EU), NSE India tick-by-tick data.

---

#### 2. Corporate Insider Transactions & Corporate Buyback Flow
* **What is missing**:
  Executive insider buying is one of the strongest empirical leading signals for equity returns over a 30-day horizon, while opportunistic corporate buyback accelerations provide price support.
* **Proposed Indicators**:
  - **Net Insider Buy Ratio (NIBR)**: Insider buy volume divided by total insider transaction volume over rolling 30-day and 90-day windows.
  - **Executive Cluster Purchases**: Indicator triggered when $\ge 3$ distinct C-suite executives buy shares within 10 trading days.
  - **Share Repurchase Execution Velocity**: Quarterly buyback announcements vs. actual treasury shares retired.
* **Data Sources**:
  - US: SEC EDGAR Form 4 filings via [OpenInsider](http://openinsider.com/) or SEC EDGAR API.
  - EU: BaFin / AMF / ESMA Article 19 Market Abuse Regulation (MAR) insider registries.
  - India: NSE / BSE Insider Trading Disclosures (SEBI PIT Regulations).

---

#### 3. Financial NLP Sentiment & Narrative Drift
* **What is missing**:
  Earnings conference calls, 8-K / 6-K material event filings, and executive tone shifts convey forward-looking information months before it fully reflects in backward-looking macro indicators or moving averages.
* **Proposed Indicators**:
  - **FinBERT Management Tone Score**: Differential sentiment score between Management Discussion & Analysis (MD&A) and Analyst Q&A segments.
  - **Guidance Surprise Embeddings**: Semantic cosine distance between consecutive earnings conference call outlook statements.
  - **News Narrative Shock Index**: Real-time media sentiment z-score from Bloomberg and Reuters financial wires.
* **Data Sources**:
  - [Alpha Vantage](https://www.alphavantage.co/) News Sentiment API.
  - SEC EDGAR 10-Q / 10-K / 8-K raw transcript parsers via huggingface `yiyanghkust/finbert-tone`.

---

#### 4. Options Market Implied Distributions (Forward-Looking Pricing)
* **What is missing**:
  Equities and technical indicators reflect past trades. Options contracts reflect forward-looking risk-neutral probability distributions priced by institutional hedgers.
* **Proposed Indicators**:
  - **25-Delta Option Skew**: Implied volatility spread between out-of-the-money puts and calls ($IV_{put} - IV_{call}$), measuring downside tail protection demand.
  - **Option Volatility Surface Term Structure**: Ratio of 30-day IV to 90-day IV to gauge impending event risk.
  - **Gamma Exposure (GEX)**: Market maker hedging positioning—positive GEX acts as a volatility dampener, while negative GEX accelerates trending moves.
* **Data Sources**:
  - [CBOE DataShop](https://datashop.cboe.com/) / [OptionMetrics IvyDB](https://optionmetrics.com/).
  - [Tradier API](https://tradier.com/) or NSE India equity derivatives option chain feed.

---

#### 5. Macro Plumbing & Global Liquidity Indicators
* **What is missing**:
  The current dataset includes FRED yields (`DGS10`, `DGS5`, `FEDFUNDS`), but misses actual balance sheet liquidity and reserve dynamics:
* **Proposed Indicators**:
  - **Net Federal Reserve Liquidity**: $\text{Fed Total Assets} - \text{Treasury General Account (TGA)} - \text{Overnight Reverse Repo (ON RRP)}$.
  - **Cross-Currency Basis Spreads (EUR/USD, JPY/USD)**: Measures global dollar funding strains affecting multinational large-caps.
  - **Credit Default Swap (CDS) Spreads & High Yield OAS**: Leading indicator of credit cycle contractions that precede stock drawdowns.
* **Data Sources**:
  - [FRED (Federal Reserve Bank of St. Louis)](https://fred.stlouisfed.org/) via `pandas_datareader`.
  - BIS (Bank for International Settlements) global liquidity statistics.